# Setup

## Imports

In [1]:
import os
import sys
import json
import time
import shutil
import subprocess
import importlib
import nltk
import pandas as pd
import torch
from transformers import GPT2TokenizerFast

from google.colab import drive

## Environment

Installs what Colab does not have: pycocoevalcap and inflect for scoring, Java for the COCO tokenizer and METEOR, NLTK data for CHAIR, and the ClipCap repo for the model class. Rerun every new session.

In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pycocoevalcap', 'inflect'], check=True)

if shutil.which('java') is None:
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'default-jre'], check=True)

print(subprocess.run(['java', '-version'], capture_output=True, text=True).stderr.split('\n')[0])

for pkg in ['punkt', 'punkt_tab', 'wordnet', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)

# ClipCap model code; train.py imports AdamW from transformers, which newer versions removed
CLIP_ROOT = '/content/CLIP_prefix_caption'
if not os.path.isdir(CLIP_ROOT):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/rmokady/CLIP_prefix_caption.git', CLIP_ROOT], check=True)

train_py = os.path.join(CLIP_ROOT, 'train.py')

with open(train_py) as fh:
    src = fh.read()

old = 'from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW, get_linear_schedule_with_warmup'
new = ('from transformers import GPT2Tokenizer, GPT2LMHeadModel, get_linear_schedule_with_warmup\n'
       'from torch.optim import AdamW')

if old in src:
    with open(train_py, 'w') as fh:
        fh.write(src.replace(old, new))
    print('patched ClipCap train.py')

if CLIP_ROOT not in sys.path:
    sys.path.append(CLIP_ROOT)

openjdk version "21.0.12.1" 2026-08-18
patched ClipCap train.py


## Paths and Configuration

SPLIT picks val or test. Keep it on val. Test is run once, for the final configs only, and needs CONFIRM_TEST = True as a guard against an accidental test run.

In [3]:
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/sc4001_nndl_grp_project'

# run settings
SPLIT = 'test'
CONFIRM_TEST = True
SEEDS = [0, 1, 2]
BATCH_SIZE = 64
USE_SPICE = False

# final configs
FINAL_RUNS = [
    'objswap_structured_lam0_m0.2_transformer_p40',
    'objswap_structured_lam3_m0.2_transformer_p40',
    'objswap_random_lam3_m0.2_transformer_p40',
    'objswap_structured_ul_lam3_transformer_p40'
]

# on val only the check run is decoded, to compare with Phase 2
VAL_CHECK_RUNS = ['objswap_structured_lam0_m0.2_transformer_p40_seed0']

# tolerances for the val check, in points
CHECK_TOL = {
    'CIDEr': 0.2,
    'CHAIRi': 0.1,
    'CHAIRs': 0.1
}

if SPLIT not in ['val', 'test']:
    raise ValueError('SPLIT must be val or test')

if SPLIT == 'test' and not CONFIRM_TEST:
    raise RuntimeError('SPLIT is test but CONFIRM_TEST is False. Only run test once, after the code freeze.')

# notebook folder
TRACK_DIR = os.path.join(PROJECT_DIR, 'phase_3', 'test_eval')
if not os.path.isdir(TRACK_DIR):
    raise FileNotFoundError(TRACK_DIR + ' not found. Create it on Drive or add the project shortcut to My Drive.')

# inputs
TRACK1_DIR = os.path.join(PROJECT_DIR, 'phase_1', 'track_1')
DATA_DIR = os.path.join(TRACK1_DIR, 'data')
EVAL_DIR = os.path.join(TRACK1_DIR, 'eval')
FEATURES_PATH = os.path.join(TRACK1_DIR, 'features', 'coco_rn50x4_' + SPLIT + '.pkl')
P2_DIR = os.path.join(PROJECT_DIR, 'phase_2', 'track_1_and_3')
P2_MODELS = os.path.join(P2_DIR, 'out', 'models')
P2_RESULTS = os.path.join(P2_DIR, 'out', 'results')
CODE_FILES = ['track_2.py', 'track1_eval.py']

for path in [os.path.join(DATA_DIR, 'karpathy_index.json'), os.path.join(EVAL_DIR, 'chair_gt_' + SPLIT + '.pkl'), os.path.join(EVAL_DIR, 'synonyms.txt'), FEATURES_PATH, P2_MODELS, P2_RESULTS]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

# outputs
RESULTS_DIR = os.path.join(TRACK_DIR, 'results')
CACHE_DIR = os.path.join(TRACK_DIR, 'cache')
TABLES_DIR = os.path.join(TRACK_DIR, 'tables')

for d in [RESULTS_DIR, CACHE_DIR, TABLES_DIR]:
    if os.path.isdir(d):
        print('exists ', d)
    else:
        os.makedirs(d)
        print('created', d)

PROGRESS_PATH = os.path.join(CACHE_DIR, 'progress_' + SPLIT + '.json')

# local copy temporarily
CODE_DIR = '/content/phase3_code'

DEVICE = 'cpu'
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU: decoding 5000 images on CPU is very slow. Runtime > Change runtime type > T4 GPU.')

print('split:', SPLIT)

Mounted at /content/drive
exists  /content/drive/MyDrive/sc4001_nndl_grp_project/phase_3/test_eval/results
exists  /content/drive/MyDrive/sc4001_nndl_grp_project/phase_3/test_eval/cache
exists  /content/drive/MyDrive/sc4001_nndl_grp_project/phase_3/test_eval/tables
GPU: Tesla T4
split: test


## Load Project Code

Uses the team's code unchanged, so captions and scores are produced exactly as in Phase 2:
- track_2.py: model construction and greedy decoding (max 30 tokens, stop at the first full stop).
- track1_eval.py: BLEU-4, METEOR, CIDEr, CHAIRi, CHAIRs, caption length and object recall, plus the per-image file.

track1_eval reads its folders from environment variables when it is imported, so they are set first. Its results folder is this notebook's results folder.

In [4]:
if not os.path.isdir(CODE_DIR):
    os.makedirs(CODE_DIR)
for name in CODE_FILES:
    shutil.copy(os.path.join(P2_DIR, name), os.path.join(CODE_DIR, name))
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.environ['TRACK2_DATA_DIR'] = DATA_DIR
os.environ['TRACK2_EVAL_DIR'] = EVAL_DIR
os.environ['TRACK2_RESULTS_DIR'] = RESULTS_DIR
os.environ['JAVA_TOOL_OPTIONS'] = '-Xlog:disable'

import track_2
import track1_eval
importlib.reload(track_2)
importlib.reload(track1_eval)

TOKENIZER = GPT2TokenizerFast.from_pretrained('gpt2')
SPLIT_DATA = track_2.load_features(FEATURES_PATH)
print('features:', SPLIT_DATA['features'].shape[0], SPLIT, 'images')

categories: 80 object words: 403
helpers ready
index loaded: 123287 images
evaluation helpers ready
categories: 80 object words: 403
helpers ready
index loaded: 123287 images
evaluation helpers ready


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

features: 5000 test images


# Generate and Evaluate

## Method

For each run, the trained mapping network is loaded from its Phase 2 checkpoint on top of frozen GPT-2. Captions for every image in the split are generated with greedy decoding, up to 30 tokens, the same protocol as Phase 2. The captions are then scored with Track 1's evaluate(), which also writes the per-image file used by the analysis notebook.

Files are named like Phase 2, with the split at the end, for example clipcap_objswap_structured_lam3_m0.2_transformer_p40_seed0_test.json, .eval.json and .per_image.json.

Save and resume: decoding saves partial captions as it goes, evaluate() skips a run whose eval file exists, and a progress file on Drive records each finished run, so a disconnected session continues where it stopped.

## Helper Functions

In [5]:
def checkpoint_path(run):
    return os.path.join(P2_MODELS, 'clipcap_' + run + '_mapping.pt')

def caption_path(run):
    return os.path.join(RESULTS_DIR, 'clipcap_' + run + '_' + SPLIT + '.json')

def load_model(run):
    # checkpoint holds the run config and the mapping network weights; GPT-2 comes from the hub
    saved = torch.load(checkpoint_path(run), map_location='cpu')
    model = track_2.model_from_config(saved['config'])
    model.clip_project.load_state_dict(saved['clip_project'])
    return model.to(DEVICE).eval()

def runs_for_split():
    # val: the check run only
    # test: every final config and seed
    if SPLIT == 'val':
        return list(VAL_CHECK_RUNS)
    runs = []
    for stem in FINAL_RUNS:
        for seed in SEEDS:
            runs.append(stem + '_seed' + str(seed))
    return runs

def generate_and_evaluate(run):
    start = time.time()
    model = load_model(run)
    track_2.decode_split(model, TOKENIZER, SPLIT_DATA, caption_path(run), DEVICE, batch_size=BATCH_SIZE)
    del model

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    result = track1_eval.evaluate(caption_path(run), SPLIT, spice=USE_SPICE)

    return {
        'scores': result['scores'],
        'minutes': round((time.time() - start) / 60, 1)
    }

def read_progress():
    if os.path.exists(PROGRESS_PATH):
        with open(PROGRESS_PATH) as fh:
            return json.load(fh)
    return {}

def write_progress(progress):
    # tmp then rename, so a disconnect never leaves half a file
    with open(PROGRESS_PATH + '.tmp', 'w') as fh:
        json.dump(progress, fh, indent=1)
    os.replace(PROGRESS_PATH + '.tmp', PROGRESS_PATH)

## Run

In [6]:
# load checkpoint, start empty if none
progress = read_progress()

for run in runs_for_split():
    if run in progress:
        print('done already:', run)
        continue

    if not os.path.exists(checkpoint_path(run)):
        print('skipped, no checkpoint yet:', run)
        continue

    print('\n' + run)
    progress[run] = generate_and_evaluate(run)
    write_progress(progress)
    print('saved progress,', progress[run]['minutes'], 'min')

print('\nfinished', len(progress), 'runs on', SPLIT)


objswap_structured_lam0_m0.2_transformer_p40_seed0


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

captioning:   0%|          | 0/79 [00:00<?, ?it/s]

evaluating clipcap_objswap_structured_ul_lam3_transformer_p40_seed1_test on test
  coverage ok: 5000 images
  tokenising...
  BLEU...
  METEOR...
  CIDEr...
  CHAIR...
  saved clipcap_objswap_structured_ul_lam3_transformer_p40_seed1_test.eval.json and clipcap_objswap_structured_ul_lam3_transformer_p40_seed1_test.per_image.json
----------------------------------------
file : clipcap_objswap_structured_ul_lam3_transformer_p40_seed1_test
split: test  images: 5000
----------------------------------------
BLEU-4          0.3102
METEOR          0.2637
CIDEr           1.0527
CHAIRi          0.0687
CHAIRs          0.1044
avg_len         9.7128
obj_recall      0.5585
----------------------------------------
saved progress, 8.5 min

objswap_structured_ul_lam3_transformer_p40_seed2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

captioning:   0%|          | 0/79 [00:00<?, ?it/s]

evaluating clipcap_objswap_structured_ul_lam3_transformer_p40_seed2_test on test
  coverage ok: 5000 images
  tokenising...
  BLEU...
  METEOR...
  CIDEr...
  CHAIR...
  saved clipcap_objswap_structured_ul_lam3_transformer_p40_seed2_test.eval.json and clipcap_objswap_structured_ul_lam3_transformer_p40_seed2_test.per_image.json
----------------------------------------
file : clipcap_objswap_structured_ul_lam3_transformer_p40_seed2_test
split: test  images: 5000
----------------------------------------
BLEU-4          0.3072
METEOR          0.2625
CIDEr           1.0453
CHAIRi          0.0706
CHAIRs          0.1050
avg_len         9.8252
obj_recall      0.5588
----------------------------------------
saved progress, 8.6 min

finished 12 runs on test


# Verify Against Phase 2

## Method

Only meaningful on val. The check run is decoded again here and compared with the val results Phase 2 already produced for the same checkpoint. Two checks:
- Caption match: the share of images whose caption is identical in both files. Greedy decoding is deterministic, so this should be close to 100%. A few differences can come from GPU rounding (Phase 2 ran on a V100, Colab uses a T4).
- Scores: each metric must be within CHECK_TOL points of Phase 2.

If both pass, this notebook reproduces Phase 2 and its test numbers can be trusted.

## Helper Functions

In [7]:
def caption_match(run):
    # share of images with the identical caption in this notebook and in Phase 2
    with open(caption_path(run)) as fh:
        ours = json.load(fh)

    with open(os.path.join(P2_RESULTS, 'clipcap_' + run + '_val.json')) as fh:
        theirs = json.load(fh)

    by_id = {}

    for rec in theirs:
        by_id[rec['image_id']] = rec['caption']

    same = 0

    for rec in ours:
        if by_id.get(rec['image_id']) == rec['caption']:
            same += 1

    return 100.0 * same / len(ours)

def score_check(run):
    # rows of metric, ours, Phase 2, difference and pass, all in points
    with open(os.path.join(RESULTS_DIR, 'clipcap_' + run + '_val.eval.json')) as fh:
        ours = json.load(fh)['scores']

    with open(os.path.join(P2_RESULTS, 'clipcap_' + run + '_val.eval.json')) as fh:
        theirs = json.load(fh)['scores']

    rows = []

    for metric in CHECK_TOL:
        a = ours[metric] * 100
        b = theirs[metric] * 100

        rows.append({
            'metric': metric,
            'this notebook': round(a, 2),
            'phase 2': round(b, 2),
            'diff': round(a - b, 2),
            'pass': abs(a - b) <= CHECK_TOL[metric]
        })

    return rows

## Compare

In [11]:
if SPLIT != 'val':
    print('Verification only runs on val. Set SPLIT = val to rerun the check (passed: 99.98% identical captions, scores match Phase 2).')
else:
    all_pass = True

    for run in VAL_CHECK_RUNS:
        if not os.path.exists(caption_path(run)):
            print('not generated yet:', run)
            all_pass = False
            continue

        match = caption_match(run)
        print(run)
        print('  identical captions:', format(match, '.2f') + '%')
        check_df = pd.DataFrame(score_check(run))
        print(check_df.to_string(index=False))

        for ok in check_df['pass']:
            if not ok:
                all_pass = False

    if all_pass:
        print('\nPASS: this notebook reproduces Phase 2. Safe to run on test after the freeze.')
    else:
        print('\nCHECK FAILED: do not run on test until this is fixed.')

Verification only runs on val. Set SPLIT = val to rerun the check (passed: 99.98% identical captions, scores match Phase 2).


# Summary

## Results Table

One row per run on the current split, written to tables/ as a CSV. The analysis notebook reads the .eval.json and .per_image.json files from results/ for the full analysis.

In [10]:
progress = read_progress()
rows = []

for run in runs_for_split():
    if run not in progress:
        continue

    scores = progress[run]['scores']
    row = {'run': run}

    for metric in ['BLEU-4', 'METEOR', 'CIDEr', 'SPICE', 'CHAIRi', 'CHAIRs', 'obj_recall']:
        if metric in scores:
            row[metric] = round(scores[metric] * 100, 2)

    row['avg_len'] = round(scores['avg_len'], 2)
    row['minutes'] = progress[run]['minutes']
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(os.path.join(TABLES_DIR, 'runs_' + SPLIT + '.csv'), index=False)
summary_df

,run,BLEU-4,METEOR,CIDEr,CHAIRi,CHAIRs,obj_recall,avg_len,minutes
0,objswap_structured_lam0_m0.2_transformer_p40_s...,30.94,26.40,104.72,7.12,10.64,56.08,9.79,7.7
1,objswap_structured_lam0_m0.2_transformer_p40_s...,31.14,26.55,105.21,6.37,9.54,56.14,9.82,8.9
2,objswap_structured_lam0_m0.2_transformer_p40_s...,30.57,26.41,104.25,6.92,10.48,55.83,9.83,8.0
3,objswap_structured_lam3_m0.2_transformer_p40_s...,30.35,26.36,103.95,7.15,10.70,56.05,9.84,8.3
4,objswap_structured_lam3_m0.2_transformer_p40_s...,31.22,26.52,105.66,6.71,9.96,56.01,9.80,8.2
5,objswap_structured_lam3_m0.2_transformer_p40_s...,30.49,26.28,103.94,6.68,10.04,55.57,9.87,8.7
6,objswap_random_lam3_m0.2_transformer_p40_seed0,30.30,26.31,103.62,6.47,9.88,55.80,9.80,8.7
7,objswap_random_lam3_m0.2_transformer_p40_seed1,30.96,26.54,105.48,6.93,10.48,56.07,9.77,8.2
8,objswap_random_lam3_m0.2_transformer_p40_seed2,31.46,26.67,106.16,6.44,9.66,55.72,9.80,8.0
9,objswap_structured_ul_lam3_transformer_p40_seed0,31.18,26.47,105.26,6.83,10.32,55.85,9.79,8.5
